# 04 - Feature Selection (Simple MI Pipeline)

Step 1: Variance Threshold + Mutual Information on genes

Step 2: Domain-specific feature engineering (7 features)

Step 3: Combine selected genes + engineered + clinical features

All logic lives in `src/feature_selection.py`. This notebook only loads data, runs the pipeline, and saves artifacts.

In [1]:
import sys
from pathlib import Path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import joblib
import config
from src.io import logger
from src.feature_selection import run_feature_selection

## Step 1: Load Preprocessed Training Data

In [2]:
X_train = pd.read_csv(config.PROCESSED_DIR / "X_train_preprocessed.csv", index_col=0)

y_train_df = pd.read_csv(config.PROCESSED_DIR / "y_train.csv")
y_train = y_train_df.iloc[:, 0] if len(y_train_df.columns) == 1 else y_train_df["BCR"]

logger.info(f"Training data loaded: {X_train.shape}, Positives: {int(y_train.sum())}")

2026-09-01 01:08:49 | INFO     | prostate_bcr | Training data loaded: (343, 19018), Positives: 46


## Step 2: Run Feature Selection Pipeline

This performs:
1. Variance threshold on genes
2. Mutual Information to select top K genes
3. Creates 7 engineered features
4. Returns fitted selector + list of all selected features

In [3]:
config.MODELS_DIR.mkdir(parents=True, exist_ok=True)
config.TABLES_DIR.mkdir(parents=True, exist_ok=True)

# Run feature selection
fitted_selector, selected_genes = run_feature_selection(
    X=X_train,
    y=y_train,
    variance_threshold=config.VARIANCE_THRESHOLD,
    mi_top_k=config.MI_TOP_K,
    random_state=config.RANDOM_STATE
)

# Create engineered features on training data
from src.feature_selection import create_engineered_features
X_eng, eng_features = create_engineered_features(X_train, selected_genes=selected_genes)

# Final features = selected genes + engineered features + clinical columns
clinical_cols = fitted_selector["clinical_cols"]
final_features = selected_genes + eng_features + clinical_cols

# Remove duplicates while preserving order
seen = set()
unique_features = []
for f in final_features:
    if f not in seen:
        seen.add(f)
        unique_features.append(f)

final_features = unique_features

print(f"Selected genes:        {len(selected_genes)}")
print(f"Engineered features:   {len(eng_features)}")
print(f"Clinical features:     {len(clinical_cols)}")
print(f"Total final features:  {len(final_features)}")

## Step 3: Save Artifacts

In [4]:
# Save fitted selector
joblib.dump(fitted_selector, config.MODELS_DIR / "fitted_selector.joblib")
print(f"Saved selector to: {config.MODELS_DIR / 'fitted_selector.joblib'}")

# Save feature list
pd.DataFrame({"feature": final_features}).to_csv(
    config.TABLES_DIR / "selected_features_final.csv", index=False
)
print(f"Saved feature list ({len(final_features)} items) to: {config.TABLES_DIR / 'selected_features_final.csv'}")

# Save engineered training data for reference
X_train_eng = X_train.copy()
for feat in eng_features:
    if feat not in X_train_eng.columns:
        X_train_eng[feat] = X_eng[feat]

print("Feature selection complete!")